1. Inspect user/item features
        
2. Convert categorical/numerical features
        
3. Build positive user-item pairs
        
4. Split train/test positives
        
5. Negative sampling
        
6. User Tower
        
7. Item Tower
        
8. Two-Tower + BCE
        
9. Precompute item embeddings
        
10. FAISS

11. Evaluating metrics
     
12. Filter already-interacted movies

In [258]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [259]:
# 1. Load User Demographics
user_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']

users = pd.read_csv(
    'ml-100k/u.user', 
    sep='|', 
    names=user_cols, 
    engine='python'
)

# 2. Load Item Features
genre_cols = [
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 
    'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 
    'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

movie_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL'] + genre_cols

movie = pd.read_csv(
    'ml-100k/u.item', 
    sep='|', 
    names=movie_cols, 
    encoding='latin-1', 
    engine='python'
).drop(columns=['video_release_date', 'IMDb_URL'])

# 3. Load Ratings (u.data - tab-separated)
rating_cols = ['user_id', 'movie_id', 'rating', 'timestamp']

ratings = pd.read_csv(
    'ml-100k/u.data', 
    sep='\t', 
    names=rating_cols, 
    engine='python'
)

In [260]:
print(ratings.head())

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596


In [261]:
# Note that this move could break if IDs are not contiguous
ratings["movie_id"] = ratings['movie_id'] - 1 # For 0 idx
ratings["user_id"] = ratings['user_id'] - 1 # For 0 idx

In [262]:
num_users = ratings['user_id'].nunique()
num_items = ratings['movie_id'].nunique()

In [263]:
pos_df = ratings[ratings['rating']>=4]
pos_df['label']=1
pos_df = pos_df.drop(columns='rating')
pos_df.head()

,user_id,movie_id,timestamp,label
5,297,473,884182806,1
7,252,464,891628467,1
11,285,1013,879781125,1
12,199,221,876042340,1
16,121,386,879270459,1


In [264]:
pos_df.shape

(55375, 4)

In [265]:
# We;ll now split train_test by leave-one-out, specifically the last rating of each user
pos_df = pos_df.sort_values(["user_id", "timestamp"])
test_df = pos_df.groupby("user_id").tail(1)
train_df = pos_df.drop(test_df.index)
# notice that we don't even need sklearn train-test-split
# Our test cases will now = num_user because each user have only 1 test case. 
# We'll evaluate recall by how many users got recommeded their hid movie/ num_user

In [266]:
train_df = train_df.drop(columns='timestamp')
test_df = test_df.drop(columns='timestamp')

In [267]:
print(train_df.shape)
print(test_df.shape)

(54433, 3)
(942, 3)


## Deal with movies features

In [268]:
print(movie.head())

   movie_id        movie_title release_date  unknown  Action  Adventure  \
0         1   Toy Story (1995)  01-Jan-1995        0       0          0   
1         2   GoldenEye (1995)  01-Jan-1995        0       1          1   
2         3  Four Rooms (1995)  01-Jan-1995        0       0          0   
3         4  Get Shorty (1995)  01-Jan-1995        0       1          0   
4         5     Copycat (1995)  01-Jan-1995        0       0          0   

   Animation  Children's  Comedy  Crime  ...  Fantasy  Film-Noir  Horror  \
0          1           1       1      0  ...        0          0       0   
1          0           0       0      0  ...        0          0       0   
2          0           0       0      0  ...        0          0       0   
3          0           0       1      0  ...        0          0       0   
4          0           0       0      1  ...        0          0       0   

   Musical  Mystery  Romance  Sci-Fi  Thriller  War  Western  
0        0        0        0 

In [269]:
movie['movie_id'] = movie['movie_id']-1 # for 0 idx
print(movie.head())

   movie_id        movie_title release_date  unknown  Action  Adventure  \
0         0   Toy Story (1995)  01-Jan-1995        0       0          0   
1         1   GoldenEye (1995)  01-Jan-1995        0       1          1   
2         2  Four Rooms (1995)  01-Jan-1995        0       0          0   
3         3  Get Shorty (1995)  01-Jan-1995        0       1          0   
4         4     Copycat (1995)  01-Jan-1995        0       0          0   

   Animation  Children's  Comedy  Crime  ...  Fantasy  Film-Noir  Horror  \
0          1           1       1      0  ...        0          0       0   
1          0           0       0      0  ...        0          0       0   
2          0           0       0      0  ...        0          0       0   
3          0           0       1      0  ...        0          0       0   
4          0           0       0      1  ...        0          0       0   

   Musical  Mystery  Romance  Sci-Fi  Thriller  War  Western  
0        0        0        0 

We extract IDs since IDs is not an input of tower. But we don't throw away IDs since we need them to later find the movies in train_df

In [270]:
genre_cols = movie.columns[3:].tolist()

print(genre_cols)
print(len(genre_cols))

['unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
19


In [271]:
movie_features = movie[genre_cols].values
print(movie_features.shape)
print(movie_features[:5]) # first 5 movies' genre

(1682, 19)
[[0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 1 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0]]


## Move to user features

In [272]:
print(users.head())

   user_id  age gender  occupation zip_code
0        1   24      M  technician    85711
1        2   53      F       other    94043
2        3   23      M      writer    32067
3        4   24      M  technician    43537
4        5   33      F       other    15213


In [273]:
users['user_id'] = users['user_id']-1 # for 0 idx
print(users.head())

   user_id  age gender  occupation zip_code
0        0   24      M  technician    85711
1        1   53      F       other    94043
2        2   23      M      writer    32067
3        3   24      M  technician    43537
4        4   33      F       other    15213


In [274]:
# Encoding
users["gender_idx"] = users["gender"].map({"M": 0, "F": 1})

print(users[["gender", "gender_idx"]].head())

  gender  gender_idx
0      M           0
1      F           1
2      M           0
3      M           0
4      F           1


In [275]:
users = users.drop(columns='gender')
print(users.head())

   user_id  age  occupation zip_code  gender_idx
0        0   24  technician    85711           0
1        1   53       other    94043           1
2        2   23      writer    32067           0
3        3   24  technician    43537           0
4        4   33       other    15213           1


In [276]:
# Standard scaler on age
from sklearn.preprocessing import StandardScaler

age_scaler = StandardScaler()

users["age"] = age_scaler.fit_transform(users[["age"]])

In [277]:
# Create a new variable then concat into users df is cleaner than directly create a new col in users df (users['occ_one_hot']=...)
occupation_encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

occupation_encoded = occupation_encoder.fit_transform(
    users[["occupation"]]
)

occupation_cols = occupation_encoder.get_feature_names_out(["occupation"])

occupation_encoded_df = pd.DataFrame(
    occupation_encoded,
    columns=occupation_cols,
    index=users.index
)

users = pd.concat([users, occupation_encoded_df], axis=1)

In [278]:
users = users.drop(columns='occupation')

In [279]:
users.head()

,user_id,age,zip_code,gender_idx,occupation_administrator,occupation_artist,occupation_doctor,occupation_educator,occupation_engineer,occupation_entertainment,...,occupation_marketing,occupation_none,occupation_other,occupation_programmer,occupation_retired,occupation_salesman,occupation_scientist,occupation_student,occupation_technician,occupation_writer
0,0,-0.824859,85711,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,1.554867,94043,1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,-0.906919,32067,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,3,-0.824859,43537,0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,4,-0.086324,15213,1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [280]:
# In this specific case, we'll drop zip code for the sake of simplicity since preprocessing zip code is pretty much messy. 
# We'll have to set each unique zip code an idx since there are too many zip code to one-hot.
# Then we need to embedd those zip_idx just like we we did with UserIDs in CF.
# So even though zip code is somewhat a good feature but in this case, we'll drop it. AMEN
users = users.drop(columns='zip_code')
users.head()

,user_id,age,gender_idx,occupation_administrator,occupation_artist,occupation_doctor,occupation_educator,occupation_engineer,occupation_entertainment,occupation_executive,...,occupation_marketing,occupation_none,occupation_other,occupation_programmer,occupation_retired,occupation_salesman,occupation_scientist,occupation_student,occupation_technician,occupation_writer
0,0,-0.824859,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,1.554867,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,-0.906919,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,3,-0.824859,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,4,-0.086324,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


We extract IDs since IDs is not an input of tower. But we don't throw away IDs since we need them to later find the users in train_df

In [281]:

# This line will go last when we done creating and modifying cols.
user_feature_cols = users.columns[1:].tolist()

print(user_feature_cols)
print(len(user_feature_cols))

['age', 'gender_idx', 'occupation_administrator', 'occupation_artist', 'occupation_doctor', 'occupation_educator', 'occupation_engineer', 'occupation_entertainment', 'occupation_executive', 'occupation_healthcare', 'occupation_homemaker', 'occupation_lawyer', 'occupation_librarian', 'occupation_marketing', 'occupation_none', 'occupation_other', 'occupation_programmer', 'occupation_retired', 'occupation_salesman', 'occupation_scientist', 'occupation_student', 'occupation_technician', 'occupation_writer']
23


In [282]:
user_features = users[user_feature_cols].values

print(user_features.shape)
print(user_features[:5])
# users df will now have age as 1st col, gender as 2nd, occupation_onehot are remaining cols 

(943, 23)
[[-0.82485939  0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          1.          0.        ]
 [ 1.55486734  1.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          1.          0.          0.
   0.          0.          0.          0.          0.        ]
 [-0.90691893  0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          1.        ]
 [-0.82485939  0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.    

# Negative Sampling

In [283]:
user_positive_items = (
    pos_df
    .groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)
user_positive_items[10]

{7,
 8,
 14,
 21,
 27,
 46,
 50,
 55,
 69,
 78,
 82,
 85,
 96,
 99,
 106,
 110,
 124,
 134,
 172,
 184,
 190,
 193,
 195,
 202,
 207,
 212,
 228,
 229,
 236,
 238,
 240,
 257,
 267,
 276,
 285,
 290,
 300,
 311,
 316,
 317,
 331,
 349,
 355,
 356,
 371,
 392,
 401,
 422,
 424,
 426,
 427,
 428,
 432,
 433,
 434,
 507,
 523,
 526,
 543,
 548,
 579,
 602,
 651,
 658,
 662,
 689,
 691,
 698,
 706,
 712,
 713,
 717,
 722,
 728,
 730,
 732,
 735,
 736,
 739,
 740,
 743,
 744,
 745,
 748,
 749,
 751}

In [284]:
import random

num_negatives = 1 # Number of negative samples to generate per positive sample
train_row = []

# We will iterate through each user-item pair in the training set and generate negative samples for each positive sample. 1:1 ratio
for _, row in train_df.iterrows():
    user_id = row['user_id']
    positive_item_id = row['movie_id']

    # positive
    train_row.append([user_id, positive_item_id, 1])

    # item the user has positively interacted with
    user_positive_items_set = user_positive_items[user_id] # Note that user_positive_items is built from the entire positive_ratings, not just the training set. This is important to avoid sampling negative items that is postiviely interacted in the test set.

    # Item that are not in the user's positive interactions
    negative_candidates = list(set(range(num_items)) - user_positive_items_set)

    # sample 1 negative item
    negative_item_id = random.choice(negative_candidates)
    train_row.append([user_id, negative_item_id, 0])

train_df = pd.DataFrame(train_row, columns=['user_id', 'movie_id', 'label'])

In [285]:
# Final check before plug this into Dataloader
print(train_df.head())
print(train_df["user_id"].min(), train_df["user_id"].max())
print(train_df["movie_id"].min(), train_df["movie_id"].max())

print(user_features.shape)
print(movie_features.shape)

   user_id  movie_id  label
0        0       167      1
1        0       978      0
2        0       171      1
3        0      1674      0
4        0       164      1
0 942
0 1681
(943, 23)
(1682, 19)


In [286]:
# Final check before plug this into Dataloader
row = train_df.iloc[0]

print(user_features[row["user_id"]])
print(movie_features[row["movie_id"]])
print(row["label"])

[-0.82485939  0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          1.          0.        ]
[0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
1


In [287]:
train_df.head()

,user_id,movie_id,label
0,0,167,1
1,0,978,0
2,0,171,1
3,0,1674,0
4,0,164,1


## We'll now plug data into dataloader

In [288]:

class CBFDataset(Dataset):
    def __init__(self, df, user_features, movie_features):
        self.user_ids = torch.tensor(
            df["user_id"].values,
            dtype=torch.long
        )

        self.movie_ids = torch.tensor(
            df["movie_id"].values,
            dtype=torch.long
        )

        self.labels = torch.tensor(
            df["label"].values,
            dtype=torch.float32
        )

        self.user_features = torch.tensor(
            user_features,
            dtype=torch.float32
        )

        self.movie_features = torch.tensor(
            movie_features,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        user_id = self.user_ids[idx]
        movie_id = self.movie_ids[idx]

        # Note that the order of this return needs to match the order in train loop
        return (
            self.user_features[user_id],
            self.movie_features[movie_id],
            self.labels[idx]
        )

In [289]:
train_set = CBFDataset(train_df,user_features,movie_features)
train_loader = DataLoader(train_set,256,shuffle=True)
# Because we don't plug test_df into MLP, that's why we don't need it to go through dataset and dataloader

In [290]:
user_x, movie_x, y = next(iter(train_loader))

print(user_x.shape)
print(movie_x.shape)
print(y.shape)

torch.Size([256, 23])
torch.Size([256, 19])
torch.Size([256])


## Build Two Tower

In [291]:
class TwoTower(nn.Module):
    def __init__(self, user_input_dim, movie_input_dim, embedding_dim=32):
        super().__init__()

        self.user_mlp = nn.Sequential(nn.Linear(user_input_dim, 64),
                                    nn.ReLU(),
                                    nn.Linear(64, embedding_dim))

        self.movie_mlp = nn.Sequential(nn.Linear(movie_input_dim, 64),
                                    nn.ReLU(),
                                    nn.Linear(64, embedding_dim))

    def forward(self,user_features,movie_features):
        user_embedding = self.user_mlp(user_features)
        movie_embedding = self.movie_mlp(movie_features)

        score = (user_embedding*movie_embedding).sum(dim=1)
        return score


In [292]:
# For OOP clarity, we're just building the MLP by providing the cols of user_features and movie_features, we haven't plug anything into forward for it to compute yet
# We will plug user_features ad movie_features for it to compute later when we model(user_features,model_features)
model = TwoTower(user_input_dim=user_features.shape[1],movie_input_dim=movie_features.shape[1],embedding_dim=32)

## Set loss, optim, device

In [293]:

device = torch.device(
    "mps" if torch.mps.is_available() else "cpu"
)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

print(device)

mps


## Train loop

In [294]:
epochs = 20

for epoch in range(epochs):

    model.train()
    total_loss = 0
    # Note that the order of this train loop needs to match the order in CBFDataset class
    for batch_user_features, batch_item_features, batch_labels in train_loader:

        batch_user_features = batch_user_features.to(device)
        batch_item_features = batch_item_features.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()

        logits = model(batch_user_features, batch_item_features)

        loss = criterion(logits, batch_labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Train Loss: {avg_loss:.4f}"
    )

Epoch 1/20, Train Loss: 0.6441
Epoch 2/20, Train Loss: 0.6241
Epoch 3/20, Train Loss: 0.6184
Epoch 4/20, Train Loss: 0.6155
Epoch 5/20, Train Loss: 0.6138
Epoch 6/20, Train Loss: 0.6117
Epoch 7/20, Train Loss: 0.6111
Epoch 8/20, Train Loss: 0.6091
Epoch 9/20, Train Loss: 0.6085
Epoch 10/20, Train Loss: 0.6077
Epoch 11/20, Train Loss: 0.6070
Epoch 12/20, Train Loss: 0.6062
Epoch 13/20, Train Loss: 0.6054
Epoch 14/20, Train Loss: 0.6049
Epoch 15/20, Train Loss: 0.6045
Epoch 16/20, Train Loss: 0.6036
Epoch 17/20, Train Loss: 0.6037
Epoch 18/20, Train Loss: 0.6030
Epoch 19/20, Train Loss: 0.6028
Epoch 20/20, Train Loss: 0.6026


In [295]:
model.eval() # Turn off training mode

with torch.no_grad():

    # Generate movie embeddings
    movie_tensor = torch.tensor(
        movie_features,
        dtype=torch.float32
    ).to(device)

    movie_embeddings = model.movie_mlp(movie_tensor)

    # Generate user embeddings
    user_tensor = torch.tensor(
        user_features,
        dtype=torch.float32
    ).to(device)

    user_embeddings = model.user_mlp(user_tensor)


# Convert embeddings to NumPy
movie_embeddings_np = (
    movie_embeddings
    .cpu()
    .numpy()
    .astype("float32")
)

user_embeddings_np = (
    user_embeddings
    .cpu()
    .numpy()
    .astype("float32")
)


# Build FAISS index
import faiss

embedding_dim = movie_embeddings_np.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(movie_embeddings_np)


# Make sure the query is contiguous float32
user_embeddings_np = np.ascontiguousarray(
    user_embeddings_np,
    dtype=np.float32
)

# Use a single CPU thread
faiss.omp_set_num_threads(1)

#For each user, we recommend all movies in score descending order. 
# Why? Because we'll take top_k later when we evaluate.
# If we take top_k in FAISS and then filter out interated movies in train_df, our top 10 could only have 2 left
# [A, B, C, D, E, F, G, H, I, J] but A-H are interacted movies, which means we're left with I,J after we filter our interacted movies
distances, movie_ids = index.search(
    user_embeddings_np,
    num_items
)

# So the matrix should be (943,1682). The order of rows start from user 0,1,2,3,...
print(movie_ids.shape) 
print(movie_ids[0]) # User 0 ALL movies in score descending order

(943, 1682)
[ 180   49  171 ... 1075 1291  547]


In [296]:
import numpy as np

K_values = [10, 50, 100, 200, 500, 1000]

# Recompute all_top_k for the LARGEST K once, then slice smaller Ks from it
max_K = max(K_values)

# ratings includes EVERY interaction (rating 1-5), including each user's held-out
# test movie (test_df is a subset of pos_df, which is a subset of ratings).
# If we filter using seen_movies_by_user as-is, the test movie can never appear
# in all_top_k, so recall@K would be 0 by construction, not by model performance.
# Fix: remove each user's held-out test movie from their "seen" set so it stays
# eligible to be retrieved.
test_movie_by_user = test_df.set_index("user_id")["movie_id"].to_dict()

seen_movies_by_user = (
    ratings
    .groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)

for user_id, test_movie_id in test_movie_by_user.items():
    seen_movies_by_user[user_id].discard(test_movie_id)

all_top_k = []
for user_id in range(num_users):
    ranked_movies = movie_ids[user_id]
    seen_movies = seen_movies_by_user.get(user_id, set())
    filtered_movies = [m for m in ranked_movies if m not in seen_movies]
    all_top_k.append(filtered_movies[:max_K])

results = {}

for K in K_values:
    hits = 0
    precision_sum = 0
    ndcg_sum = 0
    num_eval_users = 0

    for user_id in range(num_users):
        test_movie_id = test_movie_by_user.get(user_id)
        if test_movie_id is None:
            continue

        num_eval_users += 1
        top_k = all_top_k[user_id][:K]

        if test_movie_id in top_k:
            hits += 1
            rank = top_k.index(test_movie_id)
            precision_sum += 1 / K
            ndcg_sum += 1 / np.log2(rank + 2)

    results[K] = {
        "recall": hits / num_eval_users,
        "precision": precision_sum / num_eval_users,
        "ndcg": ndcg_sum / num_eval_users,
    }

for K, metrics in results.items():
    print(f"K={K:>4} | Recall: {metrics['recall']:.4f} | "
        f"Precision: {metrics['precision']:.4f} | NDCG: {metrics['ndcg']:.4f}")

K=  10 | Recall: 0.0499 | Precision: 0.0050 | NDCG: 0.0271
K=  50 | Recall: 0.1242 | Precision: 0.0025 | NDCG: 0.0435
K= 100 | Recall: 0.2091 | Precision: 0.0021 | NDCG: 0.0571
K= 200 | Recall: 0.3323 | Precision: 0.0017 | NDCG: 0.0743
K= 500 | Recall: 0.5870 | Precision: 0.0012 | NDCG: 0.1049
K=1000 | Recall: 0.8110 | Precision: 0.0008 | NDCG: 0.1284


In [297]:
all_top_k[0] 
# This give us top k NON-INTERACTED (not even interaction where rating 1,2,3,4,5) movies of user 0. 
# But still have the held-out test case since we wanna see if it can retrieve it

[np.int64(407),
 np.int64(384),
 np.int64(1005),
 np.int64(428),
 np.int64(421),
 np.int64(473),
 np.int64(630),
 np.int64(769),
 np.int64(301),
 np.int64(1484),
 np.int64(548),
 np.int64(482),
 np.int64(285),
 np.int64(802),
 np.int64(650),
 np.int64(514),
 np.int64(325),
 np.int64(306),
 np.int64(664),
 np.int64(422),
 np.int64(1283),
 np.int64(1100),
 np.int64(586),
 np.int64(361),
 np.int64(654),
 np.int64(896),
 np.int64(449),
 np.int64(448),
 np.int64(379),
 np.int64(372),
 np.int64(1585),
 np.int64(1477),
 np.int64(805),
 np.int64(1533),
 np.int64(1495),
 np.int64(481),
 np.int64(524),
 np.int64(483),
 np.int64(497),
 np.int64(1212),
 np.int64(1207),
 np.int64(1068),
 np.int64(648),
 np.int64(347),
 np.int64(331),
 np.int64(328),
 np.int64(1088),
 np.int64(750),
 np.int64(747),
 np.int64(567),
 np.int64(1482),
 np.int64(312),
 np.int64(404),
 np.int64(1187),
 np.int64(434),
 np.int64(1595),
 np.int64(975),
 np.int64(915),
 np.int64(824),
 np.int64(770),
 np.int64(297),
 np.int64

## Ways to improve metrics:

Hard negative sampling instead of random sampling

1:4 pos:neg ratio instead of 1:1

Better MLP structure

More Epochs, tuning ...
